# alternating soundsource analysis

In [ ]:
from scipy.io import loadmat
from scipy.signal import butter, filtfilt
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
from workbench.data.preprocess import TriggRasterPY, combTableCreate, processTableRow
import h5py
import tables as tb

In [ ]:
conn = sqlite3.connect(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
sql = """
SELECT * FROM Recordings WHERE (Animal_Id, Cell_Id) IN (
SELECT Animal_Id, Cell_Id FROM Recordings WHERE Condition IN ("Baseline", "soso") GROUP BY Animal_Id, Cell_Id 
HAVING COUNT(DISTINCT Condition) >= 2) 
AND Condition IN ("Baseline","soso") AND use = 1 AND Folders_generated = 1
"""

datatable= pd.read_sql_query(sql, conn)
conn.close()

In [ ]:
# compute the preprocessed data from the rawdata
#datapath = r"\\172.25.250.112\burgalossi\lab share\Data\Florian\comb_tables"
#filename = "soso_comb.5"
#try:
#    print('hi')
#except:
#    Warning("comb_table does not exist. Creating the file....")
#    comb_table = combTableCreate(datatable, datapath, filename)


In [ ]:
d_path = r"\\172.25.250.112\burgalossi\lab share\Data\Florian\ADN\FH8Soso\analysis\Data8\Baseline\exp_data.mat"
exp = loadmat(
    d_path,
    struct_as_record=False,
    squeeze_me=False,
    simplify_cells= True  
)

In [ ]:
processed_data = exp.get('processed_data', None)
raw_data = exp.get('raw_data', None)
spkT = processed_data['spike_sorting_data']['spike_times']
videoT = raw_data['ephys_data']['ttl_times']
angles = processed_data['tracking_data']['angles']

In [ ]:
import polars as pl
print(datatable.keys())
pl_df = pl.from_dataframe(datatable[['Animal_Id', 'Cell_Id', 'Condition', 'exp_type', 'Folderpath']])

In [ ]:
a = []
for o in pl_df.iter_rows(named=True):
    a.append(processTableRow(o))